In [15]:
import pandas as pd

categorical_cols = [
    "시술 당시 나이", "시술 유형", "배아 생성 주요 이유",
    "난자 출처", "정자 출처", "난자 기증자 나이", "정자 기증자 나이",
    "시술_인공수정방식"
]

train = pd.read_csv('train_preprocessed_ff.csv')
test = pd.read_csv('test_preprocessed_ff.csv')

# Int64 -> 문자열 -> category 순서로 변환 (XGBoost가 카테고리 값으로 문자열을 요구함)
for col in categorical_cols:
    train[col] = train[col].astype('Int64').astype(str).replace('<NA>', pd.NA).astype('category')
    test[col] = test[col].astype('Int64').astype(str).replace('<NA>', pd.NA).astype('category')

print('Train:', train.shape, '/ Test:', test.shape)

Train: (256351, 69) / Test: (90067, 68)


In [16]:
from sklearn.model_selection import train_test_split

target = '임신 성공 여부'
X = train.drop(columns=[target])
y = train[target]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [17]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score

model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    enable_categorical=True,
    tree_method='hist',
    random_state=42,
    n_estimators=500
)

model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

valid_pred = model.predict_proba(X_valid)[:, 1]
print('Validation ROC-AUC:', roc_auc_score(y_valid, valid_pred))

Validation ROC-AUC: 0.7242635802970998


In [18]:
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance.head(15))

             feature  importance
2              시술 유형    0.569421
37          이식된 배아 수    0.145357
48             난자 출처    0.032524
39          저장된 배아 수    0.016465
0           시술 당시 나이    0.010908
60         배아 이식 경과일    0.010515
5   착상 전 유전 검사 사용 여부    0.008916
53       신선 배아 사용 여부    0.007326
34         총 생성 배아 수    0.005931
33          DI 출산 횟수    0.005584
49             정자 출처    0.005560
28           총 임신 횟수    0.005141
50         난자 기증자 나이    0.005114
31           총 출산 횟수    0.004939
29         IVF 임신 횟수    0.004573


In [20]:
test_pred = model.predict_proba(test)[:, 1]

test_id = pd.read_csv('test.csv')['ID']  # 원본에서 ID 가져오기

result = pd.DataFrame({'ID': test_id, 'xgb_pred': test_pred})
result.to_csv('xgb_predictions.csv', index=False)
print('저장 완료:', result.shape)

저장 완료: (90067, 2)


In [21]:
print(result.head(10))
print(result['xgb_pred'].describe())  # min, max 확인

           ID  xgb_pred
0  TEST_00000  0.000190
1  TEST_00001  0.000233
2  TEST_00002  0.135717
3  TEST_00003  0.103570
4  TEST_00004  0.449941
5  TEST_00005  0.118942
6  TEST_00006  0.435657
7  TEST_00007  0.267365
8  TEST_00008  0.238920
9  TEST_00009  0.000151
count    9.006700e+04
mean     2.569682e-01
std      1.765541e-01
min      1.746169e-08
25%      1.198981e-01
50%      2.581829e-01
75%      3.794267e-01
max      9.979542e-01
Name: xgb_pred, dtype: float64
